In [2]:
%pwd

'd:\\Muzammil\\Medical Chatbot\\medical-chatbot\\research'

In [3]:
import os
os.chdir("../")

In [4]:
%pwd

'd:\\Muzammil\\Medical Chatbot\\medical-chatbot'

In [ ]:
from langchain.document_loaders import PyPDFLoader, DirectoryLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter

In [9]:
def load_pdf_files(data):
    loader = DirectoryLoader(
        data,
        glob="*.pdf",
        loader_cls=PyPDFLoader
    )

    documents = loader.load()
    return documents

In [10]:
extracted_data = load_pdf_files("data")

In [12]:
len(extracted_data)

82

In [70]:
from typing import List
from langchain.schema import Document

def filter_to_minimal_docs(docs: List[Document]) -> List[Document]:
    minimal_docs = []
    for doc in docs:
        src_path = doc.metadata.get("source")
        minimal_docs.append(
            Document(
                page_content=doc.page_content,
                metadata={"source": src_path}
            )
        )
    return minimal_docs

In [16]:
minimal_docs = filter_to_minimal_docs(extracted_data)

In [17]:
minimal_docs

[Document(metadata={'source': 'data\\2286-brain-aneurysms.pdf'}, page_content='Brain aneurysm \nDepartment of Neurosurgery\nPatient information\nUniversity \nHospitals Sussex\nNHS Foundation Trust'),
 Document(metadata={'source': 'data\\2286-brain-aneurysms.pdf'}, page_content='2\nThis leaflet is for patients, their families and carers to provide \ninformation about brain aneurysms, including incidental finding, \nrisk factors for having an aneurysm, tests and investigations, and \nthe treatment options and management of the condition.\nWhat is a brain aneurysm?\nA brain aneurysm (sometimes called a cerebral aneurysm) is  \na weakness in the wall of the blood vessel (artery) in the brain.  \nThis is a bulge which is similar to a balloon-like swelling on the artery. \nThis balloon-like swelling may remain stable or may grow or rupture.'),
 Document(metadata={'source': 'data\\2286-brain-aneurysms.pdf'}, page_content='3\nWhat is a Subarachnoid Haemorrhage?\nA brain aneurysm can rupture wi

In [18]:
def text_split(minimal_docs):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=500,
        chunk_overlap=20
    )
    text_chunks = text_splitter.split_documents(minimal_docs)
    return text_chunks

In [20]:
text_chunks = text_split(minimal_docs)
print(f"Number of text chunks: {len(text_chunks)}")

Number of text chunks: 253


In [21]:
from langchain.embeddings import HuggingFaceEmbeddings

def download_embeddings():
    embeddings = HuggingFaceEmbeddings(
        model_name="sentence-transformers/all-MiniLM-L6-v2"
    )
    return embeddings

embeddings = download_embeddings()

C:\Users\muzshf\AppData\Local\Temp\ipykernel_61284\2168566139.py:4: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(
c:\Users\muzshf\AppData\Local\anaconda3\envs\mcb\lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\muzshf\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_

In [22]:
embeddings

HuggingFaceEmbeddings(client=SentenceTransformer(
  (0): Transformer({'max_seq_length': 256, 'do_lower_case': False}) with Transformer model: BertModel 
  (1): Pooling({'word_embedding_dimension': 384, 'pooling_mode_cls_token': False, 'pooling_mode_mean_tokens': True, 'pooling_mode_max_tokens': False, 'pooling_mode_mean_sqrt_len_tokens': False, 'pooling_mode_weightedmean_tokens': False, 'pooling_mode_lasttoken': False, 'include_prompt': True})
  (2): Normalize()
), model_name='sentence-transformers/all-MiniLM-L6-v2', cache_folder=None, model_kwargs={}, encode_kwargs={}, multi_process=False, show_progress=False)

In [24]:
vec = embeddings.embed_query("This is a sample document")
print(len(vec))

384


In [31]:
from dotenv import load_dotenv
from pathlib import Path
import os

# Explicit .env path
env_path = Path.cwd() / ".env"
print(Path.cwd())
print(env_path)
load_dotenv(env_path)

PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

if PINECONE_API_KEY is None:
    raise RuntimeError("PINECONE_API_KEY not found. Check .env path.")

if OPENAI_API_KEY is None:
    raise RuntimeError("OPENAI_API_KEY not found. Check .env path.")

print("Keys loaded successfully ✅")

d:\Muzammil\Medical Chatbot\medical-chatbot
d:\Muzammil\Medical Chatbot\medical-chatbot\.env
Keys loaded successfully ✅


In [ ]:
from pinecone import Pinecone
pinecone_api_key = PINECONE_API_KEY

pc = Pinecone(api_key=pinecone_api_key)

In [37]:
from pinecone import ServerlessSpec
index_name = "medical-chatbot"
if not pc.has_index(index_name):
    pc.create_index(
        name=index_name,
        dimension=384,
        metric="cosine",
        spec=ServerlessSpec(cloud="aws", region="us-east-1")
    )

index = pc.Index(index_name)

In [39]:
from langchain_pinecone import PineconeVectorStore

docsearch = PineconeVectorStore.from_documents(
    documents=text_chunks,
    embedding=embeddings,
    index_name=index_name
)

In [40]:
# ADD more documents/files

from langchain.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter

# Load ONE PDF
loader = PyPDFLoader("data/brain_aneurysm_book.pdf")
documents = loader.load()

minimal = filter_to_minimal_docs(documents)

text_chunks = text_split(minimal)

# Add to vector DB
docsearch.add_documents(text_chunks)

print(f"Added {len(text_chunks)} chunks from one PDF")


Added 2073 chunks from one PDF


In [43]:
retriever = docsearch.as_retriever(search_type="similarity", search_kwargs={"k": 3})
retrieved_docs = retriever.invoke("What is a brain aneurysm?")
retrieved_docs

[Document(id='54f79a3e-6f95-4982-9fe7-a38b2785ba9d', metadata={'source': 'data\\cerebral-aneurysms.pdf'}, page_content='1\nCerebral Aneurysms\nWhat is a cerebral aneurysm?\nA \ncerebral aneurysm (also known as a \nbrain aneurysm) is a weak or thin spot \non an artery in the brain that balloons or \nbulges out and fills with blood. The bulging \naneurysm can put pressure on the nerves or \nbrain tissue. It may also burst or rupture, \nspilling blood into the surrounding tissue \n(called a hemorrhage). A ruptured aneurysm \ncan cause serious health problems such as \nhemorrhagic stroke, brain damage, coma, \nand even death.'),
 Document(id='c13ac56f-63b6-4991-9fda-7d1c8f213395', metadata={'source': 'data\\Detection-and-Treatment-Guide.pdf'}, page_content='BRAIN ANEURYSMS\nDETECTION AND TREATMENT'),
 Document(id='9b737281-0b60-40a7-a66a-8ec371593fdc', metadata={'source': 'data\\2286-brain-aneurysms.pdf'}, page_content='2\nThis leaflet is for patients, their families and carers to provide 

In [ ]:
# from langchain_openai import ChatOpenAI

# chat_model = ChatOpenAI(model_name="gpt-4o")

In [57]:
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

from langchain_community.chat_models import ChatOllama
# from langchain_ollama import ChatOllama

chat_model = ChatOllama(
    model="llama3",   # or mistral / phi3
)

In [67]:
system_prompt = (
    "You are a Medical assistant for brain aneurysm question-answering tasks. "
    "Use the following pieces of retrieved context to answer "
    "the questions. If you don't know the answer, say that you "
    "don't know. Write long detailed answers.\n\n"
    "\n\n"
    "{context}"
)

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        ("human", "Answer the question: {input}"),
    ]
)

In [68]:
question_answering_chain = create_stuff_documents_chain(chat_model, prompt)
rag_chain = create_retrieval_chain(retriever, question_answering_chain)

In [69]:
response = rag_chain.invoke({"input": "What is a brain aneurysm?"})
print(response["answer"])

A brain aneurysm, also referred to as a cerebral aneurysm, is a weakness in the wall of the blood vessel (artery) in the brain. This anomaly results in a bulge or balloon-like swelling on the artery. In essence, it's a thin or weak spot on an artery within the brain that protrudes outward and fills with blood. The bulging aneurysm can exert pressure on nearby nerves or brain tissue, potentially leading to serious health issues if left untreated.
